# Loan Default Risk — Bulk Prediction Model
### One unified pipeline: training and inference follow the exact same path

**Key principle:** The function `preprocess()` is defined once and called in two places:
1. During training — to build `X_train` and `X_test`
2. During prediction — on any new dataset

Nothing is ever recalculated on new data. All statistics (medians, scaler mean/scale) are frozen from training and reused.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Imports
# ════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import joblib
import json
import os

from sklearn.model_selection    import train_test_split
from sklearn.preprocessing      import StandardScaler
from sklearn.linear_model       import LogisticRegression
from sklearn.metrics            import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix, classification_report
)

os.makedirs('model_artifacts', exist_ok=True)
print('Imports done ✅')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Configuration (single place to change settings)
# ════════════════════════════════════════════════════════════

# File
DATA_FILE  = 'dataset1.csv'
THRESHOLD  = 0.30          # default decision threshold
TEST_SIZE  = 0.20
RANDOM_STATE = 42

# These are the only columns the model cares about from raw data
RAW_NUMERIC_COLS = [
    'Age', 'AnnualIncome', 'MonthlyIncome', 'LoanAmount',
    'LoanTenureMonths', 'InterestRate', 'CollateralValue',
    'CreditScore', 'PastDefaults', 'NumOpenAccounts',
    'BusinessRevenue', 'ProfitMargin', 'BusinessYears'
]
RAW_CAT_COLS = ['ApplicantType', 'EmploymentType', 'LoanType']
ID_COLS      = ['CustomerID', 'LoanID']
DROP_COLS    = ['ProbDefault']  # leakage column

# Business columns that have missing values — filled with training medians
BIZ_COLS = ['BusinessRevenue', 'ProfitMargin', 'BusinessYears']

# Categorical encoding:
#   pd.get_dummies(drop_first=True) drops the FIRST category alphabetically
#   These are the base (dropped) categories — they map to all-zero vectors
CAT_BASES = {
    'ApplicantType' : 'Business',           # other: Personal
    'EmploymentType': 'Business-owner',     # others: Salaried, Self-employed, Unemployed
    'LoanType'      : 'Business Expansion'  # others: Car, Debt Consolidation, etc.
}

print('Configuration loaded ✅')
print(f'  Data file  : {DATA_FILE}')
print(f'  Threshold  : {THRESHOLD}')
print(f'  Test size  : {TEST_SIZE}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — THE ONE PREPROCESSING FUNCTION
#
# This is the single source of truth.
# Called identically during training AND prediction.
#
# Parameters
# ----------
# raw_df         : raw CSV as a DataFrame (any columns, any order)
# training_medians: dict of {col: median_value} — from training data only
# feature_columns : list of 29 column names in training order
# drop_target     : if True, remove the Default column from output
#
# What it does
# ------------
# Step 1  Fill missing BusinessRevenue / ProfitMargin / BusinessYears
#          using TRAINING medians (never recomputed on new data)
# Step 2  Drop leakage, ID, and target columns
# Step 3  Apply log1p to AnnualIncome and LoanAmount
# Step 4  Create LoanToIncome = LoanAmount / (AnnualIncome + 1)
# Step 5  One-hot encode ApplicantType, EmploymentType, LoanType
# Step 6  Add any missing training columns (value = 0)
# Step 7  Remove any extra columns not in training
# Step 8  Reorder columns to match training order exactly
# ════════════════════════════════════════════════════════════

def preprocess(raw_df, training_medians, feature_columns, drop_target=True):
    """
    Unified preprocessing pipeline.
    Identical for training data and new prediction data.
    Returns a DataFrame with exactly the columns in feature_columns.
    """
    df = raw_df.copy()

    # ── Step 1: Fill missing business columns ──────────────────────────
    # Always use TRAINING medians — never recompute on new data
    for col, val in training_medians.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)   # fill existing column
        else:
            df[col] = val                   # column absent → create it

    # ── Step 2: Drop leakage / ID / target columns ──────────────────────
    cols_to_drop = ID_COLS + DROP_COLS
    if drop_target:
        cols_to_drop.append('Default')
    df = df.drop(columns=cols_to_drop, errors='ignore')

    # ── Step 3: Log transforms ──────────────────────────────────────────
    df['Log_AnnualIncome'] = np.log1p(df['AnnualIncome'])
    df['Log_LoanAmount']   = np.log1p(df['LoanAmount'])

    # ── Step 4: LoanToIncome ratio ──────────────────────────────────────
    df['LoanToIncome'] = df['LoanAmount'] / (df['AnnualIncome'] + 1)

    # ── Step 5: One-hot encoding ─────────────────────────────────────────
    # drop_first=True ensures same base categories as training:
    #   ApplicantType  → base = Business
    #   EmploymentType → base = Business-owner
    #   LoanType       → base = Business Expansion
    df = pd.get_dummies(df, drop_first=True)

    # ── Steps 6-8: Align to training columns ────────────────────────────
    # Add missing columns with 0
    for col in feature_columns:
        if col not in df.columns:
            df[col] = 0
    # Remove extra columns + reorder to exact training order
    df = df[feature_columns]

    return df


print('preprocess() function defined ✅')
print()
print('This function will be called:')
print('  → During training  (CELL 7)')
print('  → During inference (CELL 14)')
print('  → In the UI (same logic, JavaScript version)')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Load raw dataset
# ════════════════════════════════════════════════════════════
raw = pd.read_csv(DATA_FILE)

print('Raw dataset loaded ✅')
print('Shape :', raw.shape)
print('Columns:', raw.columns.tolist())
print()
print('Missing values:')
miss = raw.isnull().sum()
print(miss[miss > 0])
print()
print('Categories:')
for col in RAW_CAT_COLS:
    print(f'  {col}: {sorted(raw[col].unique())}')
raw.head()

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Compute and LOCK training statistics
#
# Computed ONCE from training data.
# Saved immediately. Never recomputed for new data.
# ════════════════════════════════════════════════════════════

# ── Training medians (for missing business columns) ──────────
TRAINING_MEDIANS = {}
for col in BIZ_COLS:
    TRAINING_MEDIANS[col] = float(raw[col].median())

print('Training medians (computed from raw training data, LOCKED):')
for col, val in TRAINING_MEDIANS.items():
    missing_count = raw[col].isnull().sum()
    print(f'  {col}: {val}  ({missing_count} missing rows will be filled)')

# Save immediately
joblib.dump(TRAINING_MEDIANS, 'model_artifacts/training_medians.pkl')
print()
print('Saved → model_artifacts/training_medians.pkl ✅')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — Create target variable
# ════════════════════════════════════════════════════════════
raw['Default'] = (raw['ProbDefault'] > THRESHOLD).astype(int)

print('Target variable created ✅')
print(f'  Rule: Default = 1  if ProbDefault > {THRESHOLD}')
print()
print('Distribution:')
print(raw['Default'].value_counts())
print(f'  Default rate: {raw["Default"].mean()*100:.2f}%')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — Preprocess for training
#
# Calls the SAME preprocess() function defined in CELL 3.
# No other preprocessing happens anywhere else.
# ════════════════════════════════════════════════════════════

# We need feature_columns before we can call preprocess().
# So we run it once to discover the columns, then save them.

# Temporarily remove Default so we can discover feature columns
raw_no_target = raw.drop(columns=['Default'])
raw_no_target['Default'] = raw['Default']  # re-add at end

# First pass: get columns (use None as placeholder for feature_columns)
df_temp = raw.copy()
for col, val in TRAINING_MEDIANS.items():
    df_temp[col] = df_temp[col].fillna(val)
df_temp = df_temp.drop(columns=ID_COLS + DROP_COLS, errors='ignore')
df_temp['Log_AnnualIncome'] = np.log1p(df_temp['AnnualIncome'])
df_temp['Log_LoanAmount']   = np.log1p(df_temp['LoanAmount'])
df_temp['LoanToIncome']     = df_temp['LoanAmount'] / (df_temp['AnnualIncome'] + 1)
df_temp = pd.get_dummies(df_temp, drop_first=True)

# Lock feature columns (exclude target)
FEATURE_COLUMNS = [c for c in df_temp.columns if c != 'Default']

# Save immediately
joblib.dump(FEATURE_COLUMNS, 'model_artifacts/feature_columns.pkl')

print('Feature columns discovered and LOCKED ✅')
print(f'  Total features: {len(FEATURE_COLUMNS)}')
print()
for i, c in enumerate(FEATURE_COLUMNS):
    print(f'  [{i:02d}] {c}')
print()
print('Saved → model_artifacts/feature_columns.pkl ✅')

# Now run preprocess() properly
X_full = preprocess(raw, TRAINING_MEDIANS, FEATURE_COLUMNS, drop_target=True)
y_full = raw['Default']

print()
print('preprocess() complete ✅')
print('X shape:', X_full.shape)
print('y shape:', y_full.shape)
X_full.head()

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — Train / test split
# ════════════════════════════════════════════════════════════
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print('Train / test split ✅')
print(f'  Train : {X_train.shape}  defaults: {y_train.sum()}')
print(f'  Test  : {X_test.shape}   defaults: {y_test.sum()}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — StandardScaler (fit on TRAINING set only)
#
# fit_transform() → called ONCE on X_train
# transform()     → called on X_test and ALL future data
# ════════════════════════════════════════════════════════════
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform
X_test_sc  = scaler.transform(X_test)         # transform only

# Save immediately — these mean/scale values are now frozen
joblib.dump(scaler, 'model_artifacts/scaler.pkl')

print('StandardScaler fitted on X_train only ✅')
print()
print('Frozen statistics (first 5 features):')
for i in range(5):
    print(f'  [{i}] {FEATURE_COLUMNS[i]:<25}  mean={scaler.mean_[i]:>14.4f}  scale={scaler.scale_[i]:>14.4f}')
print()
print('Saved → model_artifacts/scaler.pkl ✅')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — Train model
# ════════════════════════════════════════════════════════════
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_sc, y_train)

# Save immediately
joblib.dump(model, 'model_artifacts/model.pkl')

print('LogisticRegression trained ✅')
print()
print('Top 5 features by absolute coefficient:')
coef_df = pd.DataFrame({'Feature': FEATURE_COLUMNS, 'Coef': model.coef_[0]})
top5 = coef_df.reindex(coef_df['Coef'].abs().nlargest(5).index)
for _, row in top5.iterrows():
    direction = 'RISK ↑' if row['Coef'] > 0 else 'RISK ↓'
    print(f'  {row["Feature"]:<30}  coef={row["Coef"]:>8.4f}  {direction}')
print()
print('Saved → model_artifacts/model.pkl ✅')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11 — Evaluate
# ════════════════════════════════════════════════════════════
y_pred = model.predict(X_test_sc)
y_prob = model.predict_proba(X_test_sc)[:, 1]

print('Model Evaluation\n')
print(f'  Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision : {precision_score(y_test, y_pred):.4f}')
print(f'  Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'  F1 Score  : {f1_score(y_test, y_pred):.4f}')
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print()
print('Classification Report:')
print(classification_report(y_test, y_pred))
print(f'Probability range on test set: {y_prob.min():.4f} – {y_prob.max():.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 12 — Save COMPLETE pipeline config as JSON
#
# This file contains everything needed to reproduce predictions
# in any environment (Python, JavaScript, etc.):
#   - feature_columns  : exact 29 column names and order
#   - training_medians : frozen fill values for biz columns
#   - scaler_mean/scale: frozen StandardScaler statistics
#   - intercept/coef   : exact LR weights
#   - categorical_info : which categories exist, which are bases
# ════════════════════════════════════════════════════════════
pipeline_config = {
    'feature_columns'  : FEATURE_COLUMNS,
    'training_medians' : TRAINING_MEDIANS,
    'scaler_mean'      : scaler.mean_.tolist(),
    'scaler_scale'     : scaler.scale_.tolist(),
    'intercept'        : float(model.intercept_[0]),
    'coef'             : model.coef_[0].tolist(),
    'threshold'        : THRESHOLD,
    'categorical_bases': CAT_BASES,
    'all_categories'   : {
        'ApplicantType' : sorted(raw['ApplicantType'].unique().tolist()),
        'EmploymentType': sorted(raw['EmploymentType'].unique().tolist()),
        'LoanType'      : sorted(raw['LoanType'].unique().tolist())
    }
}

with open('model_artifacts/pipeline_config.json', 'w') as f:
    json.dump(pipeline_config, f, indent=2)

print('pipeline_config.json saved ✅')
print()
print('Contents:')
print(f'  feature_columns   : {len(pipeline_config["feature_columns"])} columns')
print(f'  training_medians  : {pipeline_config["training_medians"]}')
print(f'  intercept         : {pipeline_config["intercept"]:.6f}')
print(f'  threshold         : {pipeline_config["threshold"]}')
print(f'  categorical_bases : {pipeline_config["categorical_bases"]}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 13 — Risk factor explanation (helper function)
# ════════════════════════════════════════════════════════════
def get_risk_factors(row):
    """
    Return up to 3 human-readable risk factors for one applicant.
    Works on raw (unprocessed) row data.
    """
    factors = []
    credit  = row.get('CreditScore',     np.nan)
    past_d  = row.get('PastDefaults',    0)
    ann_inc = row.get('AnnualIncome',    1) or 1
    loan    = row.get('LoanAmount',      0) or 0
    int_r   = row.get('InterestRate',    0) or 0
    collat  = row.get('CollateralValue', 0) or 0
    open_ac = row.get('NumOpenAccounts', 0) or 0
    emp     = str(row.get('EmploymentType', '')).lower()
    ltype   = str(row.get('LoanType',       '')).lower()
    lti     = loan / (ann_inc + 1)

    if not pd.isna(credit):
        if   credit < 580:  factors.append((f'Very low credit score ({int(credit)})',  2.2))
        elif credit < 650:  factors.append((f'Low credit score ({int(credit)})',        1.3))

    if past_d > 0:          factors.append((f'{int(past_d)} previous default(s)',       1.9 * past_d))

    if lti > 1.2:           factors.append((f'High LTI ratio ({lti:.2f}x)',            1.6))
    elif lti > 0.7:         factors.append((f'Elevated LTI ratio ({lti:.2f}x)',        0.9))

    if int_r > 16:          factors.append((f'High interest rate ({int_r:.1f}%)',       1.0))

    if collat > 0 and loan > 0:
        ltv = loan / collat
        if ltv > 0.85:      factors.append((f'High LTV ({ltv*100:.0f}%)',              0.9))

    if open_ac > 5:         factors.append((f'{int(open_ac)} open accounts',           0.6))
    if 'unemployed' in emp: factors.append(('Unemployed',                              0.7))
    elif 'self' in emp:     factors.append(('Self-employed',                           0.3))
    if 'debt' in ltype:     factors.append(('Debt consolidation loan',                 0.4))

    factors.sort(key=lambda x: -x[1])
    return ' | '.join(f[0] for f in factors[:3]) if factors else 'General risk profile'

print('get_risk_factors() defined ✅')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 14 — BULK PREDICTION ON ANY NEW DATASET
#
# Change NEW_FILE to any CSV path.
# The function loads the frozen artifacts and calls preprocess()
# with EXACTLY the same parameters as training.
# ════════════════════════════════════════════════════════════

# ── CONFIG: change only this ──────────────────────────────
NEW_FILE  = 'dataset1.csv'   # ← your new CSV file
PRED_THRESHOLD = 0.30        # ← decision threshold
# ─────────────────────────────────────────────────────────

def predict_on_new_data(csv_path, threshold=0.30):
    """
    End-to-end prediction on any new CSV file.

    Uses the EXACT same pipeline as training:
      preprocess() → frozen scaler.transform() → model.predict_proba()

    Tolerates:
      - Missing values (filled with training medians)
      - Missing columns (added with value 0)
      - Extra columns (ignored)
      - Different row count
    """
    # Load frozen artifacts
    _model    = joblib.load('model_artifacts/model.pkl')
    _scaler   = joblib.load('model_artifacts/scaler.pkl')
    _features = joblib.load('model_artifacts/feature_columns.pkl')
    _medians  = joblib.load('model_artifacts/training_medians.pkl')

    # Load new data
    new_raw = pd.read_csv(csv_path)
    print(f'New data loaded: {new_raw.shape}')

    # Preserve ID columns for results table
    id_data = {}
    for col in ['CustomerID', 'LoanID']:
        if col in new_raw.columns:
            id_data[col] = new_raw[col].values

    # ── PREPROCESS: same function, same parameters ────────────
    X_new = preprocess(new_raw, _medians, _features, drop_target=True)
    print(f'After preprocess: {X_new.shape}  (expected: (n, {len(_features)}))')

    # ── SCALE: frozen scaler, transform only ─────────────────
    X_new_sc = _scaler.transform(X_new)

    # ── PREDICT ───────────────────────────────────────────────
    probs = _model.predict_proba(X_new_sc)[:, 1]
    preds = (probs >= threshold).astype(int)

    # ── BUILD RESULTS TABLE ───────────────────────────────────
    results = new_raw.copy()
    results.insert(0, 'RowNumber',     np.arange(1, len(new_raw) + 1))
    results['DefaultProb_%'] = (probs * 100).round(2)
    results['Prediction']    = preds
    results['RiskLevel']     = np.where(probs > 0.65, 'High',
                               np.where(probs > threshold, 'Medium', 'Low'))
    results['TopRiskFactors'] = [
        get_risk_factors(row) if preds[i] == 1 else '—'
        for i, row in enumerate(new_raw.to_dict('records'))
    ]

    # Defaulters only, sorted by probability
    defaulters = (
        results[results['Prediction'] == 1]
        .sort_values('DefaultProb_%', ascending=False)
        .reset_index(drop=True)
    )
    return results, defaulters, probs


# ── Run ───────────────────────────────────────────────────
all_results, defaulters_df, probs = predict_on_new_data(NEW_FILE, PRED_THRESHOLD)

total    = len(all_results)
n_def    = len(defaulters_df)

print()
print('━' * 48)
print('         BULK PREDICTION SUMMARY')
print('━' * 48)
print(f'  File              : {NEW_FILE}')
print(f'  Total applicants  : {total:,}')
print(f'  Predicted default : {n_def:,}')
print(f'  Default rate      : {n_def/total*100:.2f}%')
print(f'  Prob range        : {probs.min():.4f} – {probs.max():.4f}')
print(f'  High risk (>65%)  : {(probs>0.65).sum():,}')
print('━' * 48)

# Preview
show_cols = ['RowNumber','CustomerID','LoanID','DefaultProb_%',
             'RiskLevel','TopRiskFactors','CreditScore','LoanAmount']
show_cols = [c for c in show_cols if c in defaulters_df.columns]
defaulters_df[show_cols].head(10)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 15 — Save results to CSV
# ════════════════════════════════════════════════════════════
OUTPUT_FILE = 'defaulters_output.csv'

save_cols = ['RowNumber','CustomerID','LoanID','DefaultProb_%','RiskLevel',
             'TopRiskFactors','CreditScore','LoanAmount','AnnualIncome',
             'InterestRate','PastDefaults','EmploymentType','LoanType']
save_cols = [c for c in save_cols if c in defaulters_df.columns]
defaulters_df[save_cols].to_csv(OUTPUT_FILE, index=False)

print(f'Defaulters saved → {OUTPUT_FILE} ✅')
print(f'Total defaulters : {len(defaulters_df):,}')
print()
print('Risk level breakdown:')
print(defaulters_df['RiskLevel'].value_counts().to_string())

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 16 — Pipeline integrity check
#
# Verifies that the preprocessing function produces identical
# output when called on the same data twice.
# If both runs match → pipeline is deterministic ✅
# ════════════════════════════════════════════════════════════
X1 = preprocess(raw, TRAINING_MEDIANS, FEATURE_COLUMNS)
X2 = preprocess(raw, TRAINING_MEDIANS, FEATURE_COLUMNS)

identical = (X1.values == X2.values).all()
print('Pipeline determinism check:', '✅ PASS' if identical else '❌ FAIL')
print(f'  Run 1 shape: {X1.shape}')
print(f'  Run 2 shape: {X2.shape}')
print(f'  All values identical: {identical}')
print()

# Verify column order
print('Column order matches feature_columns.pkl:')
col_match = X1.columns.tolist() == FEATURE_COLUMNS
print('  ', '✅ Match' if col_match else '❌ Mismatch')
print()
print('All artifacts summary:')
for fname in ['model.pkl','scaler.pkl','feature_columns.pkl',
              'training_medians.pkl','pipeline_config.json']:
    path = f'model_artifacts/{fname}'
    size = os.path.getsize(path) if os.path.exists(path) else 0
    status = '✅' if size > 0 else '❌'
    print(f'  {status} {fname}  ({size:,} bytes)')